# Kimlik (Identity) Fine-Tune — Gemma 3 1B

`unsloth/gemma-3-1b-it` modelini kendi kimlik veri setimle eğitiyorum. Amaç: modele
kendi adını (**Ada**) ve yaratıcısını (**Nur Şima Akgül**) öğretmek.

Notlar:
- Runtime → GPU (A100 / L4)
- İki veri dosyası (`identity_turkish_ada.jsonl`, `identity_english_ada.jsonl`) Drive'da
- `HF_TOKEN` Colab Secrets'ta (write yetkili)
- Gemma erişimini bir kez onaylamak gerekiyor

## 1) Kurulum

In [ ]:
!pip install -q unsloth
!pip install -q --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

## 2) Hugging Face girişi

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 3) Modeli yükle

In [ ]:
from unsloth import FastModel
import torch

MODEL_ID = "unsloth/gemma-3-1b-it"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    full_finetuning = False,
)
print("Model yuklendi:", MODEL_ID)

## 4) LoRA ekle

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA eklendi.")

## 5) Kimlik veri setini Drive'dan yükle (Türkçe + İngilizce)

`TR_PATH` ve `EN_PATH` kendi Drive konumuma göre.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json

TR_PATH = "/content/drive/MyDrive/identity_turkish_ada.jsonl"
EN_PATH = "/content/drive/MyDrive/identity_english_ada.jsonl"

for p in (TR_PATH, EN_PATH):
    assert os.path.exists(p), f"Dosya bulunamadi: {p} — yolu kontrol et"

records = []
for p in (TR_PATH, EN_PATH):
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

print("Toplam kayit (TR + EN):", len(records))
print("Ornek soru :", records[0][0]["content"][:80])
print("Ornek cevap:", records[0][1]["content"][:100])

## 6) Sohbet formatına çevir

In [ ]:
from datasets import Dataset

def format_ornek(rec):
    user_msg = rec[0].get("content") or ""
    asst_msg = rec[1].get("content") or ""
    text = (
        "<start_of_turn>user\n" + user_msg + "<end_of_turn>\n"
        "<start_of_turn>model\n" + asst_msg + "<end_of_turn>\n"
    )
    return {"text": text}

temiz = [r for r in records if (r[0].get("content") and r[1].get("content"))]
data = [format_ornek(r) for r in temiz]
dataset = Dataset.from_list(data).shuffle(seed=42)

print(dataset)
print(dataset[0]["text"][:400])

## 7) Eğitim

Veri küçük olduğu için 3 epoch veriyorum.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        max_length = MAX_SEQ_LENGTH,
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
trainer_stats = trainer.train()

## 8) Kimliği test et

In [ ]:
def uret(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, do_sample=True,
        repetition_penalty=1.2, no_repeat_ngram_size=3,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("SORU: Sen kimsin?")
print(uret("Sen kimsin?"))
print("\n" + "="*60 + "\n")
print("SORU: Seni kim geliştirdi?")
print(uret("Seni kim geliştirdi?"))
print("\n" + "="*60 + "\n")
print("SORU: Who are you and who created you?")
print(uret("Who are you and who created you?"))

## 9) LoRA adaptörünü Hugging Face'e yükle

In [ ]:
HF_USERNAME = "nursimakgul"
REPO_NAME = "gemma-3-1b-ada-identity-lora"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

SAVE_DIR = "gemma-3-1b-ada-identity"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Kaydedildi:", SAVE_DIR)

In [ ]:
readme_content = f"""---
base_model: {MODEL_ID}
library_name: peft
license: gemma
language:
- tr
- en
tags:
- gemma
- gemma-3
- unsloth
- lora
- peft
- identity
- turkish
pipeline_tag: text-generation
---

# {REPO_NAME}

`{MODEL_ID}` temel alinarak egitilmis bir kimlik (identity) LoRA adaptoru. Bu adaptor,
modele **Ada** adini ve yaraticisi olarak **Nur Sima Akgul**'u ogretir. Unsloth ile
fine-tune edilmistir.

## Amac

Modelin kendini tutarli bir kimlikle tanitmasi: adi Ada, yaraticisi Nur Sima Akgul.
Ornek:

> Soru: "Sen kimsin?"
> Cevap: "Ben Ada, Nur Sima Akgul tarafindan gelistirilen bir dil modeliyim..."

## Egitim detaylari

- **Temel model:** {MODEL_ID}
- **Framework:** Unsloth + TRL (SFT)
- **Yontem:** LoRA (r=16, alpha=16), 4-bit (QLoRA)
- **Epoch:** 3
- **Veri:** Referans identity veri seti (alibayram/identity_finetune_magibu_q3) temel
  alinarak kimlik bilgileri degistirilmis; Turkce + Ingilizce, toplam 1600 ornek

## Kullanim

```python
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained("{REPO_ID}", load_in_4bit=True)

messages = [{{"role": "user", "content": "Sen kimsin?"}}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
```

## Not

Bu adaptor egitim/odev amaclidir. Kimlik bilgisi disindaki genel yeteneklerde temel
modelin sinirlari gecerlidir.
"""

with open(f"{SAVE_DIR}/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)
print("README yazildi.")

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(REPO_ID, repo_type="model", exist_ok=True)
upload_folder(folder_path=SAVE_DIR, repo_id=REPO_ID, repo_type="model")
print(f"\nTamamlandi! LoRA adaptoru: https://huggingface.co/{REPO_ID}")